In [7]:
# ============================================================
# COCO 2017 SSL DOWNLOAD FIX FOR GOOGLE COLAB
# ============================================================

import subprocess
import zipfile
from pathlib import Path

COCO_ROOT = Path("/content/coco")
COCO_ROOT.mkdir(parents=True, exist_ok=True)

downloads = [
    (
        "http://images.cocodataset.org/zips/val2017.zip",
        COCO_ROOT / "val2017.zip",
    ),
    (
        "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        COCO_ROOT / "annotations_trainval2017.zip",
    ),
]


def valid_zip(path):
    if not path.exists() or path.stat().st_size < 1024:
        return False

    try:
        with zipfile.ZipFile(path, "r") as archive:
            return archive.testzip() is None
    except Exception:
        return False


def download_file(url, destination):
    # Previous failed download ko remove karega
    if destination.exists() and not valid_zip(destination):
        print("Removing incomplete file:", destination.name)
        destination.unlink()

    if valid_zip(destination):
        print("Already downloaded:", destination.name)
        return

    print("\nDownloading:", url)

    wget_command = [
        "wget",
        "-c",
        "--show-progress",
        "--tries=5",
        "--timeout=60",
        "--no-check-certificate",
        "-O",
        str(destination),
        url,
    ]

    result = subprocess.run(wget_command, check=False)

    # wget fail ho to curl fallback
    if result.returncode != 0 or not valid_zip(destination):
        print("wget failed. Trying curl fallback...")

        if destination.exists() and not valid_zip(destination):
            destination.unlink()

        curl_command = [
            "curl",
            "-L",
            "-k",
            "--fail",
            "--retry",
            "5",
            "--retry-delay",
            "3",
            "-o",
            str(destination),
            url,
        ]

        subprocess.run(curl_command, check=True)

    if not valid_zip(destination):
        raise RuntimeError(
            f"Download incomplete or corrupt: {destination}"
        )

    print("Downloaded and verified:", destination.name)


for url, destination in downloads:
    download_file(url, destination)


# Extract validation images
val_zip = COCO_ROOT / "val2017.zip"

if not (COCO_ROOT / "val2017").exists():
    print("\nExtracting val2017 images...")

    with zipfile.ZipFile(val_zip, "r") as archive:
        archive.extractall(COCO_ROOT)


# Extract annotations
annotation_zip = COCO_ROOT / "annotations_trainval2017.zip"
annotation_file = (
    COCO_ROOT
    / "annotations"
    / "instances_val2017.json"
)

if not annotation_file.exists():
    print("Extracting COCO annotations...")

    with zipfile.ZipFile(annotation_zip, "r") as archive:
        archive.extractall(COCO_ROOT)


# Final validation
image_directory = COCO_ROOT / "val2017"

assert image_directory.exists(), (
    "val2017 image directory was not created."
)

assert annotation_file.exists(), (
    "instances_val2017.json was not created."
)

image_count = len(list(image_directory.glob("*.jpg")))

print("\nCOCO dataset is ready.")
print("Images directory:", image_directory)
print("Annotation file:", annotation_file)
print("Validation images found:", image_count)


Downloading: http://images.cocodataset.org/zips/val2017.zip
Downloaded and verified: val2017.zip

Downloading: http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Downloaded and verified: annotations_trainval2017.zip

Extracting val2017 images...
Extracting COCO annotations...

COCO dataset is ready.
Images directory: /content/coco/val2017
Annotation file: /content/coco/annotations/instances_val2017.json
Validation images found: 5000


In [10]:
# ============================================================
# FIX: YOLOv5 ModuleNotFoundError — ultralytics
# ============================================================

import sys
import subprocess
import importlib
import shutil
from pathlib import Path

# Required dependency install
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "ultralytics",
    "gitpython",
    "setuptools",
])

importlib.invalidate_caches()

# Verify exact import required by YOLOv5
import torch
import ultralytics
from ultralytics.utils.patches import torch_load

print("Torch version:", torch.__version__)
print("Ultralytics version:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

# Remove old/incomplete YOLOv5 Torch Hub cache
yolov5_cache = (
    Path(torch.hub.get_dir())
    / "ultralytics_yolov5_master"
)

if yolov5_cache.exists():
    shutil.rmtree(
        yolov5_cache,
        ignore_errors=True,
    )
    print("Old YOLOv5 cache removed.")

print("YOLOv5 dependency fix completed.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Torch version: 2.11.0+cpu
Ultralytics version: 8.4.102
CUDA available: False
Old YOLOv5 cache removed.
YOLOv5 dependency fix completed.


In [11]:
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

test_model = torch.hub.load(
    "ultralytics/yolov5",
    "yolov5x6",
    pretrained=True,
    trust_repo=True,
    skip_validation=True,
    force_reload=True,
)

test_model = test_model.to(device).eval()

print("YOLOv5x6 successfully loaded on:", device)
print("Model classes:", len(test_model.names))

Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip
requirements: Ultralytics requirement ["urllib3>=2.6.0 ; python_version > '3.8'"] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 580ms
Prepared 1 package in 160ms
Uninstalled 1 package in 19ms
Installed 1 package in 11ms
 - urllib3==2.5.0
 + urllib3==2.7.0

requirements: AutoUpdate success ✅ 1.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



YOLOv5 🚀 2026-7-19 Python-3.12.13 torch-2.11.0+cpu CPU

100%|██████████| 270M/270M [00:06<00:00, 41.3MB/s]

Fusing layers... 
YOLOv5x6 summary: 574 layers, 140730220 parameters, 0 gradients, 209.6 GFLOPs
Adding AutoShape... 


YOLOv5x6 successfully loaded on: cpu
Model classes: 80


In [12]:
from __future__ import annotations
import json
import math
import os
import random
import shutil
import urllib.request
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from pycocotools.coco import COCO
from sklearn.cluster import KMeans
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURATION
# ============================================================
# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

COCO_ROOT = Path("/content/coco")
COCO_IMAGE_DIR = COCO_ROOT / "val2017"
COCO_ANNOTATION_FILE = COCO_ROOT / "annotations" / "instances_val2017.json"
OUTPUT_DIR = Path("/content/rce_baseline_comparison_outputs")

AUTO_DOWNLOAD_COCO = True
MODEL_NAME = "yolov5x6"
MODEL_INPUT_SIZE = 640  # T4-safe; use 1280 on A100 for the final paper run
TARGET_RESULTS = 2  # quick validation; set 10 for the final experiment
MAX_CANDIDATES_PER_PAIR = 50
K_CONCEPTS = 6  # quick mode; set 8 for the final experiment
SOFTMAX_TEMPERATURE = 0.05
ACTIVATION_PERCENTILE = 85
BLUR_RADIUS = 15
INITIAL_DETECTION_CONF = 0.01
INTERVENTION_DETECTION_CONF = 0.01
MATCH_IOU_THRESHOLD = 0.10
SHOW_TOP_CONCEPTS = 6
SAVE_PDF = True
SAVE_COUNTERFACTUALS = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class PairSpec:
    object_a: str
    object_b: str
    max_gap_fraction: float = 0.35
    max_union_fraction: float = 0.92
    max_examples: int = 1



# سب سے عام اور آسانی سے ملنے والے COCO جوڑے
PAIR_SPECS: List[PairSpec] = [
    PairSpec("person", "laptop", max_gap_fraction=1.0, max_union_fraction=1.0, max_examples=3),
    PairSpec("dog", "frisbee", max_gap_fraction=1.0, max_union_fraction=1.0, max_examples=3),
    PairSpec("person", "horse", max_gap_fraction=1.0, max_union_fraction=1.0, max_examples=3),
    PairSpec("person", "motorcycle", max_gap_fraction=1.0, max_union_fraction=1.0, max_examples=3),
]


# ============================================================
# 2. COCO DOWNLOAD / PATH VALIDATION
# ============================================================

def _download_with_progress(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading: {url}")
    urllib.request.urlretrieve(url, destination)


def _extract_zip(zip_path: Path, destination: Path) -> None:
    print(f"Extracting: {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)


def ensure_coco_files(auto_download: bool = AUTO_DOWNLOAD_COCO) -> None:
    if COCO_IMAGE_DIR.exists() and COCO_ANNOTATION_FILE.exists():
        print("COCO 2017 validation images and annotations are available.")
        return

    if not auto_download:
        raise FileNotFoundError(
            "COCO files were not found.\n"
            f"Expected images: {COCO_IMAGE_DIR}\n"
            f"Expected annotations: {COCO_ANNOTATION_FILE}\n\n"
            "Upload/mount the dataset at these paths, or set "
            "AUTO_DOWNLOAD_COCO = True and rerun."
        )

    COCO_ROOT.mkdir(parents=True, exist_ok=True)
    image_zip = COCO_ROOT / "val2017.zip"
    annotation_zip = COCO_ROOT / "annotations_trainval2017.zip"

    if not image_zip.exists():
        _download_with_progress(
            "https://images.cocodataset.org/zips/val2017.zip", image_zip
        )
    if not annotation_zip.exists():
        _download_with_progress(
            "https://images.cocodataset.org/annotations/annotations_trainval2017.zip",
            annotation_zip,
        )

    if not COCO_IMAGE_DIR.exists():
        _extract_zip(image_zip, COCO_ROOT)
    if not COCO_ANNOTATION_FILE.exists():
        _extract_zip(annotation_zip, COCO_ROOT)

    if not (COCO_IMAGE_DIR.exists() and COCO_ANNOTATION_FILE.exists()):
        raise RuntimeError("COCO download/extraction did not create the expected files.")


# ============================================================
# 3. GEOMETRY AND IMAGE HELPERS
# ============================================================

def xywh_to_xyxy(box_xywh: Sequence[float]) -> np.ndarray:
    x, y, w, h = map(float, box_xywh)
    return np.array([x, y, x + w, y + h], dtype=np.float32)


def box_area(box: np.ndarray) -> float:
    return float(max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1]))


def union_box(box_a: np.ndarray, box_b: np.ndarray) -> np.ndarray:
    return np.array(
        [
            min(box_a[0], box_b[0]),
            min(box_a[1], box_b[1]),
            max(box_a[2], box_b[2]),
            max(box_a[3], box_b[3]),
        ],
        dtype=np.float32,
    )


def calculate_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = box_area(box_a) + box_area(box_b) - intersection
    return intersection / (union + 1e-8)


def normalized_box_gap(
    box_a: np.ndarray, box_b: np.ndarray, width: int, height: int
) -> float:
    dx = max(float(box_a[0] - box_b[2]), float(box_b[0] - box_a[2]), 0.0)
    dy = max(float(box_a[1] - box_b[3]), float(box_b[1] - box_a[3]), 0.0)
    return math.hypot(dx, dy) / (math.hypot(width, height) + 1e-8)


def normalized_center_distance(
    box_a: np.ndarray, box_b: np.ndarray, width: int, height: int
) -> float:
    center_a = np.array(
        [(box_a[0] + box_a[2]) / 2.0, (box_a[1] + box_a[3]) / 2.0]
    )
    center_b = np.array(
        [(box_b[0] + box_b[2]) / 2.0, (box_b[1] + box_b[3]) / 2.0]
    )
    return float(np.linalg.norm(center_a - center_b)) / (
        math.hypot(width, height) + 1e-8
    )


def box_mask(box: np.ndarray, size: Tuple[int, int]) -> np.ndarray:
    width, height = size
    mask = np.zeros((height, width), dtype=np.uint8)
    x1, y1, x2, y2 = np.round(box).astype(int)
    x1, x2 = np.clip([x1, x2], 0, width)
    y1, y2 = np.clip([y1, y2], 0, height)
    if x2 > x1 and y2 > y1:
        mask[y1:y2, x1:x2] = 1
    return mask


def normalize_map(values: np.ndarray, support_mask: Optional[np.ndarray] = None) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    if support_mask is None:
        support = np.ones(values.shape, dtype=bool)
    else:
        support = support_mask.astype(bool)
    result = np.zeros_like(values, dtype=np.float32)
    valid = values[support]
    if valid.size == 0:
        return result
    lo, hi = float(valid.min()), float(valid.max())
    if hi - lo <= 1e-8:
        return result
    result[support] = (values[support] - lo) / (hi - lo)
    return result


@dataclass
class LetterboxMeta:
    original_size: Tuple[int, int]      # width, height
    input_size: Tuple[int, int]         # width, height
    resized_size: Tuple[int, int]       # width, height
    ratio: float
    left: int
    top: int


def letterbox_rgb(
    image_rgb: np.ndarray,
    new_shape: Tuple[int, int] = (1280, 1280),
    color: Tuple[int, int, int] = (114, 114, 114),
) -> Tuple[np.ndarray, LetterboxMeta]:
    original_h, original_w = image_rgb.shape[:2]
    target_h, target_w = new_shape
    ratio = min(target_h / original_h, target_w / original_w)

    resized_w = int(round(original_w * ratio))
    resized_h = int(round(original_h * ratio))
    resized = cv2.resize(
        image_rgb, (resized_w, resized_h), interpolation=cv2.INTER_LINEAR
    )

    pad_w = target_w - resized_w
    pad_h = target_h - resized_h
    left = int(round(pad_w / 2.0 - 0.1))
    right = int(round(pad_w / 2.0 + 0.1))
    top = int(round(pad_h / 2.0 - 0.1))
    bottom = int(round(pad_h / 2.0 + 0.1))

    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color
    )
    meta = LetterboxMeta(
        original_size=(original_w, original_h),
        input_size=(padded.shape[1], padded.shape[0]),
        resized_size=(resized_w, resized_h),
        ratio=ratio,
        left=left,
        top=top,
    )
    return padded, meta


def map_box_to_feature(
    box_original: np.ndarray, meta: LetterboxMeta, feature_hw: Tuple[int, int]
) -> Tuple[int, int, int, int]:
    feature_h, feature_w = feature_hw
    input_w, input_h = meta.input_size
    transformed = np.array(
        [
            box_original[0] * meta.ratio + meta.left,
            box_original[1] * meta.ratio + meta.top,
            box_original[2] * meta.ratio + meta.left,
            box_original[3] * meta.ratio + meta.top,
        ],
        dtype=np.float32,
    )
    x1 = int(np.floor(transformed[0] / input_w * feature_w))
    y1 = int(np.floor(transformed[1] / input_h * feature_h))
    x2 = int(np.ceil(transformed[2] / input_w * feature_w))
    y2 = int(np.ceil(transformed[3] / input_h * feature_h))
    x1, x2 = np.clip([x1, x2], 0, feature_w)
    y1, y2 = np.clip([y1, y2], 0, feature_h)
    return int(x1), int(y1), int(x2), int(y2)


def feature_map_to_original(
    feature_map: np.ndarray, meta: LetterboxMeta
) -> np.ndarray:
    input_w, input_h = meta.input_size
    resized_w, resized_h = meta.resized_size
    original_w, original_h = meta.original_size

    upsampled = cv2.resize(
        feature_map.astype(np.float32), (input_w, input_h), interpolation=cv2.INTER_LINEAR
    )
    crop = upsampled[
        meta.top : meta.top + resized_h,
        meta.left : meta.left + resized_w,
    ]
    if crop.size == 0:
        raise RuntimeError("Feature-map crop is empty; check letterbox metadata.")
    return cv2.resize(
        crop, (original_w, original_h), interpolation=cv2.INTER_LINEAR
    )


# ============================================================
# 4. YOLOv5 DETECTION AND FEATURE EXTRACTION
# ============================================================

@dataclass
class Detection:
    box_xyxy: np.ndarray
    score: float
    label_idx: int
    class_name: str


def _collect_4d_tensors(value: Any, output: List[torch.Tensor]) -> None:
    if torch.is_tensor(value) and value.ndim == 4:
        output.append(value)
    elif isinstance(value, (list, tuple)):
        for item in value:
            _collect_4d_tensors(item, output)
    elif isinstance(value, dict):
        for item in value.values():
            _collect_4d_tensors(item, output)


class YOLOv5Processor:
    """YOLOv5 detector plus a high-resolution neck feature capture."""

    def __init__(
        self,
        model_name: str = MODEL_NAME,
        device: str = DEVICE,
        input_size: int = MODEL_INPUT_SIZE,
    ) -> None:
        self.device = device
        self.input_size = int(input_size)
        print(f"Loading {model_name} on {device}...")
        self.detector = torch.hub.load(
            "ultralytics/yolov5",
            model_name,
            pretrained=True,
            trust_repo=True,
        )
        self.detector = self.detector.to(device).eval()
        self.detector.max_det = 300

        self.backend = self.detector.model
        try:
            self.detect_layer = self.backend.model.model[-1]
        except Exception as exc:
            raise RuntimeError(
                "Could not resolve the YOLOv5 Detect layer. "
                "The torch-hub model structure may have changed."
            ) from exc
        self._captured_feature: Optional[torch.Tensor] = None

    def _capture_detect_inputs(self, module: torch.nn.Module, inputs: Tuple[Any, ...]) -> None:
        tensors: List[torch.Tensor] = []
        _collect_4d_tensors(inputs, tensors)
        if not tensors:
            self._captured_feature = None
            return
        # Detect receives multi-scale neck maps. Use the highest spatial resolution.
        selected = max(tensors, key=lambda tensor: tensor.shape[-2] * tensor.shape[-1])
        self._captured_feature = selected.detach().clone()

    def detect(
        self,
        image: Image.Image,
        conf_threshold: float = INITIAL_DETECTION_CONF,
        iou_threshold: float = 0.45,
    ) -> List[Detection]:
        self.detector.conf = float(conf_threshold)
        self.detector.iou = float(iou_threshold)
        with torch.no_grad():
            results = self.detector(image, size=self.input_size)

        detections: List[Detection] = []
        dataframe = results.pandas().xyxy[0]
        for _, row in dataframe.iterrows():
            label_idx = int(row["class"])
            detections.append(
                Detection(
                    box_xyxy=np.array(
                        [row.xmin, row.ymin, row.xmax, row.ymax],
                        dtype=np.float32,
                    ),
                    score=float(row.confidence),
                    label_idx=label_idx,
                    class_name=str(self.detector.names[label_idx]),
                )
            )
        return detections

    def extract_features(
        self, image: Image.Image
    ) -> Tuple[torch.Tensor, LetterboxMeta]:
        image_rgb = np.asarray(image.convert("RGB"))
        padded, meta = letterbox_rgb(
            image_rgb, new_shape=(self.input_size, self.input_size)
        )
        tensor = (
            torch.from_numpy(padded)
            .to(self.device)
            .permute(2, 0, 1)
            .contiguous()
            .unsqueeze(0)
            .float()
            / 255.0
        )
        if getattr(self.backend, "fp16", False):
            tensor = tensor.half()

        self._captured_feature = None
        handle = self.detect_layer.register_forward_pre_hook(
            self._capture_detect_inputs
        )
        try:
            with torch.no_grad():
                _ = self.backend(tensor, augment=False, visualize=False)
        finally:
            handle.remove()

        if self._captured_feature is None:
            raise RuntimeError("YOLOv5 feature capture failed.")
        return self._captured_feature, meta


def match_detection_to_ground_truth(
    detections: Sequence[Detection],
    class_name: str,
    ground_truth_box: np.ndarray,
    minimum_iou: float = MATCH_IOU_THRESHOLD,
) -> Optional[Detection]:
    candidates: List[Tuple[float, float, Detection]] = []
    for detection in detections:
        if detection.class_name != class_name:
            continue
        iou = calculate_iou(detection.box_xyxy, ground_truth_box)
        candidates.append((iou, detection.score, detection))
    if not candidates:
        return None
    best_iou, _, best_detection = max(candidates, key=lambda item: (item[0], item[1]))
    return best_detection if best_iou >= minimum_iou else None


# ============================================================
# 5. COCO PAIR SELECTION
# ============================================================

@dataclass
class GroundTruthPair:
    image_id: int
    file_name: str
    width: int
    height: int
    object_a: str
    object_b: str
    box_a: np.ndarray
    box_b: np.ndarray
    quality_score: float


class COCOPairSelector:
    def __init__(self, annotation_file: Path, image_dir: Path) -> None:
        self.coco = COCO(str(annotation_file))
        self.image_dir = Path(image_dir)
        categories = self.coco.loadCats(self.coco.getCatIds())
        self.category_id = {category["name"]: category["id"] for category in categories}

    def _annotations_for(
        self, image_id: int, category_name: str
    ) -> List[Dict[str, Any]]:
        category_id = self.category_id[category_name]
        annotation_ids = self.coco.getAnnIds(
            imgIds=[image_id], catIds=[category_id], iscrowd=False
        )
        annotations = self.coco.loadAnns(annotation_ids)
        return [
            annotation
            for annotation in annotations
            if float(annotation.get("area", 0.0)) > 1.0
        ]

    def candidate_pairs(
        self, spec: PairSpec, limit: int = MAX_CANDIDATES_PER_PAIR
    ) -> List[GroundTruthPair]:
        if spec.object_a not in self.category_id or spec.object_b not in self.category_id:
            return []

        image_ids = self.coco.getImgIds(
            catIds=[
                self.category_id[spec.object_a],
                self.category_id[spec.object_b],
            ]
        )
        scored: List[GroundTruthPair] = []

        for image_id in image_ids:
            image_info = self.coco.loadImgs([image_id])[0]
            width, height = int(image_info["width"]), int(image_info["height"])
            anns_a = self._annotations_for(image_id, spec.object_a)
            anns_b = self._annotations_for(image_id, spec.object_b)
            if not anns_a or not anns_b:
                continue

            best: Optional[GroundTruthPair] = None
            for annotation_a in anns_a:
                box_a = xywh_to_xyxy(annotation_a["bbox"])
                for annotation_b in anns_b:
                    box_b = xywh_to_xyxy(annotation_b["bbox"])
                    gap = normalized_box_gap(box_a, box_b, width, height)
                    pair_union = union_box(box_a, box_b)
                    union_fraction = box_area(pair_union) / float(width * height)
                    if gap > spec.max_gap_fraction:
                        continue
                    if union_fraction > spec.max_union_fraction:
                        continue

                    center_distance = normalized_center_distance(
                        box_a, box_b, width, height
                    )
                    # Lower is better. The score favors spatially compact,
                    # non-trivial object pairs without claiming a relation label.
                    quality = (
                        0.50 * gap
                        + 0.30 * center_distance
                        + 0.20 * union_fraction
                    )
                    candidate = GroundTruthPair(
                        image_id=int(image_id),
                        file_name=str(image_info["file_name"]),
                        width=width,
                        height=height,
                        object_a=spec.object_a,
                        object_b=spec.object_b,
                        box_a=box_a,
                        box_b=box_b,
                        quality_score=float(quality),
                    )
                    if best is None or candidate.quality_score < best.quality_score:
                        best = candidate

            if best is not None:
                scored.append(best)

        scored.sort(key=lambda item: item.quality_score)
        return scored[:limit]

    def image_path(self, pair: GroundTruthPair) -> Path:
        return self.image_dir / pair.file_name


# ============================================================
# 6. CONCEPT DISCOVERY AND INTERVENTIONAL SCORING
# ============================================================

@dataclass
class ConceptResult:
    concept_index: int
    effect_score: float
    original_score_a: float
    original_score_b: float
    post_score_a: float
    post_score_b: float
    mask_fraction: float
    auto_region_label: str
    activation_map: np.ndarray
    intervention_mask: np.ndarray


class KMeansConceptExplainer:
    """K-Means over all feature vectors inside the selected union region."""

    def __init__(
        self,
        number_of_concepts: int = K_CONCEPTS,
        temperature: float = SOFTMAX_TEMPERATURE,
        seed: int = SEED,
    ) -> None:
        self.number_of_concepts = int(number_of_concepts)
        self.temperature = float(temperature)
        self.seed = int(seed)

    def discover(
        self,
        feature_tensor: torch.Tensor,
        relational_box: np.ndarray,
        meta: LetterboxMeta,
    ) -> torch.Tensor:
        if feature_tensor.ndim != 4 or feature_tensor.shape[0] != 1:
            raise ValueError(
                f"Expected a [1,C,H,W] feature tensor, got {feature_tensor.shape}."
            )

        _, channels, feature_h, feature_w = feature_tensor.shape
        x1, y1, x2, y2 = map_box_to_feature(
            relational_box, meta, (feature_h, feature_w)
        )
        if x2 <= x1 or y2 <= y1:
            raise RuntimeError("The relational box maps to an empty feature region.")

        feature_grid = feature_tensor[0].permute(1, 2, 0).contiguous()
        region_vectors = feature_grid[y1:y2, x1:x2].reshape(-1, channels)
        region_vectors = region_vectors[
            torch.isfinite(region_vectors).all(dim=1)
        ]
        if region_vectors.shape[0] < 2:
            raise RuntimeError("Too few valid feature vectors for concept discovery.")

        k_final = min(self.number_of_concepts, int(region_vectors.shape[0]))
        kmeans = KMeans(
            n_clusters=k_final,
            random_state=self.seed,
            n_init=10,
        )
        centers_numpy = kmeans.fit(region_vectors.cpu().numpy()).cluster_centers_
        centers = torch.as_tensor(
            centers_numpy,
            dtype=feature_tensor.dtype,
            device=feature_tensor.device,
        )

        all_vectors = feature_grid.reshape(-1, channels)
        distances = torch.cdist(all_vectors.float(), centers.float())
        similarity = F.softmax(-distances / self.temperature, dim=1)
        activation_maps = similarity.T.reshape(k_final, feature_h, feature_w)
        return activation_maps.unsqueeze(0)


def automatic_region_label(
    activation_map: np.ndarray,
    detection_a: Detection,
    detection_b: Detection,
    pair_union: np.ndarray,
) -> str:
    support = box_mask(pair_union, (activation_map.shape[1], activation_map.shape[0]))
    normalized = normalize_map(activation_map, support)
    total = float((normalized * support).sum()) + 1e-8

    mass_a = float((normalized * box_mask(detection_a.box_xyxy, (activation_map.shape[1], activation_map.shape[0]))).sum()) / total
    mass_b = float((normalized * box_mask(detection_b.box_xyxy, (activation_map.shape[1], activation_map.shape[0]))).sum()) / total
    context_mass = max(0.0, 1.0 - min(1.0, mass_a + mass_b))

    if mass_a >= 0.25 and mass_b >= 0.25:
        return "shared pair region"
    if mass_a >= max(0.20, mass_b):
        return f"{detection_a.class_name}-dominant region"
    if mass_b >= max(0.20, mass_a):
        return f"{detection_b.class_name}-dominant region"
    if context_mass >= 0.45:
        return "shared context"
    return "mixed relational region"


class RelationalInterventionExplainer:
    def __init__(
        self,
        processor: YOLOv5Processor,
        percentile: float = ACTIVATION_PERCENTILE,
        blur_radius: float = BLUR_RADIUS,
        matching_iou: float = MATCH_IOU_THRESHOLD,
    ) -> None:
        self.processor = processor
        self.percentile = float(percentile)
        self.blur_radius = float(blur_radius)
        self.matching_iou = float(matching_iou)

    def _matched_post_score(
        self,
        detections: Sequence[Detection],
        original_detection: Detection,
    ) -> float:
        candidates: List[Tuple[float, float]] = []
        for detection in detections:
            if detection.class_name != original_detection.class_name:
                continue
            iou = calculate_iou(
                original_detection.box_xyxy, detection.box_xyxy
            )
            if iou >= self.matching_iou:
                candidates.append((iou, detection.score))
        if not candidates:
            return 0.0
        return max(candidates, key=lambda item: (item[0], item[1]))[1]

    def evaluate(
        self,
        image: Image.Image,
        activation_maps: torch.Tensor,
        meta: LetterboxMeta,
        detection_a: Detection,
        detection_b: Detection,
        pair_union: np.ndarray,
        counterfactual_dir: Optional[Path] = None,
    ) -> List[ConceptResult]:
        image_rgb = np.asarray(image.convert("RGB"))
        width, height = image.size
        union_support = box_mask(pair_union, image.size)
        blurred = np.asarray(
            image.filter(ImageFilter.GaussianBlur(radius=self.blur_radius))
        )
        original_joint = detection_a.score * detection_b.score

        results: List[ConceptResult] = []
        for concept_index in range(activation_maps.shape[1]):
            feature_map = (
                activation_maps[0, concept_index].detach().cpu().numpy()
            )
            original_map = feature_map_to_original(feature_map, meta)
            normalized = normalize_map(original_map, union_support)

            values = normalized[union_support.astype(bool)]
            if values.size == 0 or float(values.max()) <= 0.0:
                threshold = 1.0
            else:
                threshold = float(np.percentile(values, self.percentile))

            intervention_mask = (
                (normalized >= threshold) & (union_support > 0)
            ).astype(np.uint8)
            mask_fraction = float(intervention_mask.sum()) / (
                float(union_support.sum()) + 1e-8
            )

            counterfactual = np.where(
                intervention_mask[..., None] > 0,
                blurred,
                image_rgb,
            ).astype(np.uint8)
            counterfactual_image = Image.fromarray(counterfactual)

            post_detections = self.processor.detect(
                counterfactual_image,
                conf_threshold=INTERVENTION_DETECTION_CONF,
            )
            post_a = self._matched_post_score(post_detections, detection_a)
            post_b = self._matched_post_score(post_detections, detection_b)

            # ہر آبجیکٹ کے سکور میں کتنی کمی آئی؟
            drop_a = max(0.0, float(detection_a.score) - post_a)
            drop_b = max(0.0, float(detection_b.score) - post_b)

            # XAI Correlation Metric: ہم min() استعمال کریں گے تاکہ صرف وہ حصہ
            # ٹاپ سکور حاصل کرے جو دونوں آبجیکٹس (Correlated) کو متاثر کرے۔
            effect = float(min(drop_a, drop_b))

            if counterfactual_dir is not None and SAVE_COUNTERFACTUALS:
                counterfactual_dir.mkdir(parents=True, exist_ok=True)
                counterfactual_image.save(
                    counterfactual_dir / f"concept_{concept_index + 1:02d}.png"
                )

            results.append(
                ConceptResult(
                    concept_index=concept_index,
                    effect_score=float(effect),
                    original_score_a=float(detection_a.score),
                    original_score_b=float(detection_b.score),
                    post_score_a=float(post_a),
                    post_score_b=float(post_b),
                    mask_fraction=float(mask_fraction),
                    auto_region_label=automatic_region_label(
                        normalized,
                        detection_a,
                        detection_b,
                        pair_union,
                    ),
                    activation_map=normalized,
                    intervention_mask=intervention_mask,
                )
            )
        return results


# ============================================================
# 7. PAPER-READY VISUALIZATION
# ============================================================

def draw_labelled_box(
    image_rgb: np.ndarray,
    detection: Detection,
    color: Tuple[int, int, int],
) -> None:
    x1, y1, x2, y2 = np.round(detection.box_xyxy).astype(int)
    cv2.rectangle(image_rgb, (x1, y1), (x2, y2), color, 3)
    label = f"{detection.class_name} {detection.score:.2f}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, thickness = 0.65, 2
    (text_w, text_h), baseline = cv2.getTextSize(
        label, font, scale, thickness
    )
    label_top = max(0, y1 - text_h - baseline - 8)
    cv2.rectangle(
        image_rgb,
        (x1, label_top),
        (x1 + text_w + 8, label_top + text_h + baseline + 8),
        color,
        -1,
    )
    cv2.putText(
        image_rgb,
        label,
        (x1 + 4, label_top + text_h + 3),
        font,
        scale,
        (0, 0, 0),
        thickness,
        cv2.LINE_AA,
    )


def make_union_overlay(
    image_rgb: np.ndarray, pair_union: np.ndarray
) -> np.ndarray:
    result = image_rgb.copy()
    overlay = result.copy()
    x1, y1, x2, y2 = np.round(pair_union).astype(int)
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (255, 0, 255), -1)
    result = cv2.addWeighted(overlay, 0.32, result, 0.68, 0.0)
    cv2.rectangle(result, (x1, y1), (x2, y2), (255, 0, 255), 4)
    return result


def combined_top_concept_heatmap(
    concept_results: Sequence[ConceptResult],
    pair_union: np.ndarray,
    image_size: Tuple[int, int],
    top_n: int = 3,
) -> np.ndarray:
    width, height = image_size
    support = box_mask(pair_union, image_size)
    ranked = sorted(
        concept_results, key=lambda result: result.effect_score, reverse=True
    )
    selected = ranked[: max(1, min(top_n, len(ranked)))]

    positive_weights = np.array(
        [max(0.0, result.effect_score) for result in selected],
        dtype=np.float32,
    )
    if float(positive_weights.sum()) <= 1e-8:
        positive_weights = np.ones(len(selected), dtype=np.float32)
    positive_weights /= positive_weights.sum()

    heatmap = np.zeros((height, width), dtype=np.float32)
    for weight, result in zip(positive_weights, selected):
        heatmap += weight * result.activation_map

    heatmap *= support
    heatmap = normalize_map(heatmap, support)
    sigma = max(2.0, min(width, height) / 120.0)
    heatmap = cv2.GaussianBlur(heatmap, (0, 0), sigmaX=sigma, sigmaY=sigma)
    return normalize_map(heatmap, support)


def overlay_heatmap(image_rgb: np.ndarray, heatmap: np.ndarray) -> np.ndarray:
    colored_bgr = cv2.applyColorMap(
        np.uint8(np.clip(heatmap, 0.0, 1.0) * 255),
        cv2.COLORMAP_JET,
    )
    colored_rgb = cv2.cvtColor(colored_bgr, cv2.COLOR_BGR2RGB)
    nonzero = heatmap[heatmap > 0]
    if nonzero.size == 0:
        return image_rgb.copy()

    threshold = float(np.percentile(nonzero, 60))
    alpha = np.clip((heatmap - threshold) / (1.0 - threshold + 1e-8), 0.0, 1.0)
    alpha = (0.78 * alpha)[..., None]
    darkened = (image_rgb.astype(np.float32) * 0.48)
    composed = darkened * (1.0 - alpha) + colored_rgb.astype(np.float32) * alpha
    return np.uint8(np.clip(composed, 0, 255))


def create_dashboard(
    image: Image.Image,
    image_id: int,
    detection_a: Detection,
    detection_b: Detection,
    pair_union: np.ndarray,
    concept_results: Sequence[ConceptResult],
    output_stem: Path,
) -> Path:
    image_rgb = np.asarray(image.convert("RGB"))
    boxed = image_rgb.copy()
    draw_labelled_box(boxed, detection_a, (0, 255, 255))
    draw_labelled_box(boxed, detection_b, (255, 255, 0))
    union_view = make_union_overlay(image_rgb, pair_union)

    heatmap = combined_top_concept_heatmap(
        concept_results, pair_union, image.size, top_n=3
    )
    heatmap_view = overlay_heatmap(image_rgb, heatmap)

    ranked = sorted(
        concept_results, key=lambda result: result.effect_score, reverse=True
    )
    displayed = ranked[: min(SHOW_TOP_CONCEPTS, len(ranked))]
    labels = [
        f"C{result.concept_index + 1}: {result.auto_region_label}"
        for result in displayed
    ]
    scores = [result.effect_score for result in displayed]

    fig, axes = plt.subplots(2, 2, figsize=(13.2, 10.5))
    fig.suptitle(
        f"Qualitative RCE Example: {detection_a.class_name} + "
        f"{detection_b.class_name}  |  COCO image {image_id}",
        fontsize=16,
        fontweight="bold",
    )

    axes[0, 0].imshow(boxed)
    axes[0, 0].set_title("1. Interacting Objects Detected", fontweight="bold")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(union_view)
    axes[0, 1].set_title("2. Relational Analysis Region", fontweight="bold")
    axes[0, 1].axis("off")

    axes[1, 0].imshow(heatmap_view)
    axes[1, 0].set_title("3. Top-Ranked RCE Concept Overlay", fontweight="bold")
    axes[1, 0].axis("off")

    y_positions = np.arange(len(labels))
    bar_colors = ["coral" if value >= 0 else "lightgray" for value in scores]
    axes[1, 1].barh(
        y_positions,
        scores,
        color=bar_colors,
        edgecolor="black",
        linewidth=0.8,
    )
    axes[1, 1].set_yticks(y_positions)
    axes[1, 1].set_yticklabels(labels, fontsize=9)
    axes[1, 1].invert_yaxis()
    axes[1, 1].axvline(0.0, linewidth=1.0, color="black")
    axes[1, 1].set_xlabel(r"Concept-Effect Score $CE_k$")
    axes[1, 1].set_title("4. Interventional Concept Ranking", fontweight="bold")
    axes[1, 1].grid(axis="x", linestyle=":", alpha=0.35)

    for y_value, score in zip(y_positions, scores):
        offset = 0.008 * max(1e-3, max(abs(np.asarray(scores))))
        horizontal = score + offset if score >= 0 else score - offset
        alignment = "left" if score >= 0 else "right"
        axes[1, 1].text(
            horizontal,
            y_value,
            f"{score:.3f}",
            va="center",
            ha=alignment,
            fontsize=8,
        )

    fig.text(
        0.5,
        0.012,
        "Concept descriptions are automatically derived from activation "
        "overlap and are not semantic labels generated by RCE.",
        ha="center",
        fontsize=9,
    )
    plt.tight_layout(rect=[0.0, 0.035, 1.0, 0.95])

    png_path = output_stem.with_suffix(".png")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    if SAVE_PDF:
        fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    return png_path


def create_contact_sheet(
    dashboard_paths: Sequence[Path],
    output_path: Path,
    columns: int = 2,
    panel_width: int = 1700,
    margin: int = 30,
) -> Path:
    if not dashboard_paths:
        raise ValueError("No dashboard paths were supplied.")

    panels: List[Image.Image] = []
    for dashboard_path in dashboard_paths:
        panel = Image.open(dashboard_path).convert("RGB")
        scale = panel_width / float(panel.width)
        panel = panel.resize(
            (panel_width, int(round(panel.height * scale))),
            Image.Resampling.LANCZOS,
        )
        panels.append(panel)

    rows = math.ceil(len(panels) / columns)
    row_heights: List[int] = []
    for row in range(rows):
        row_panels = panels[row * columns : (row + 1) * columns]
        row_heights.append(max(panel.height for panel in row_panels))

    sheet_width = columns * panel_width + (columns + 1) * margin
    sheet_height = sum(row_heights) + (rows + 1) * margin
    sheet = Image.new("RGB", (sheet_width, sheet_height), "white")

    y = margin
    for row in range(rows):
        x = margin
        row_panels = panels[row * columns : (row + 1) * columns]
        for panel in row_panels:
            sheet.paste(panel, (x, y))
            x += panel_width + margin
        y += row_heights[row] + margin

    sheet.save(output_path, dpi=(300, 300))
    return output_path


# ============================================================
# 8. END-TO-END RUNNER
# ============================================================

def analyze_ground_truth_pair(
    processor: YOLOv5Processor,
    pair: GroundTruthPair,
    selector: COCOPairSelector,
) -> Optional[Dict[str, Any]]:
    image_path = selector.image_path(pair)
    if not image_path.exists():
        return None

    image = Image.open(image_path).convert("RGB")
    img_width, img_height = image.size
    img_area = img_width * img_height

    detections = processor.detect(
        image, conf_threshold=INITIAL_DETECTION_CONF
    )

    detection_a = match_detection_to_ground_truth(
        detections, pair.object_a, pair.box_a, minimum_iou=MATCH_IOU_THRESHOLD
    )
    detection_b = match_detection_to_ground_truth(
        detections, pair.object_b, pair.box_b, minimum_iou=MATCH_IOU_THRESHOLD
    )

    if detection_a is None or detection_b is None:
        return None

    # ==============================================================
    # NAYA IZAFA: Object Size Check (صرف بڑے اور سامنے والے آبجیکٹس کے لیے)
    # ==============================================================
    area_a = box_area(detection_a.box_xyxy)
    area_b = box_area(detection_b.box_xyxy)

    # اگر کوئی بھی آبجیکٹ پوری تصویر کے 4% یا 3% سے چھوٹا ہے، تو اسے چھوڑ دیں
    if area_a < (img_area * 0.04) or area_b < (img_area * 0.03):
        print(f"\n[Skipped] Image {pair.image_id} - Objects are too small (Background).")
        return None
    # ==============================================================

    pair_union = union_box(
        detection_a.box_xyxy, detection_b.box_xyxy
    )
    features, meta = processor.extract_features(image)
    concept_explainer = KMeansConceptExplainer()
    activation_maps = concept_explainer.discover(
        features, pair_union, meta
    )

    intervention_explainer = RelationalInterventionExplainer(processor, matching_iou=MATCH_IOU_THRESHOLD)
    example_stem = (
        f"coco_{pair.image_id}_"
        f"{pair.object_a.replace(' ', '_')}_"
        f"{pair.object_b.replace(' ', '_')}"
    )
    example_dir = OUTPUT_DIR / example_stem
    example_dir.mkdir(parents=True, exist_ok=True)

    concept_results = intervention_explainer.evaluate(
        image=image,
        activation_maps=activation_maps,
        meta=meta,
        detection_a=detection_a,
        detection_b=detection_b,
        pair_union=pair_union,
        counterfactual_dir=example_dir / "counterfactuals",
    )

    dashboard_path = create_dashboard(
        image=image,
        image_id=pair.image_id,
        detection_a=detection_a,
        detection_b=detection_b,
        pair_union=pair_union,
        concept_results=concept_results,
        output_stem=example_dir / "dashboard",
    )

    rows: List[Dict[str, Any]] = []
    for result in concept_results:
        rows.append(
            {
                "image_id": pair.image_id,
                "file_name": pair.file_name,
                "object_a": pair.object_a,
                "object_b": pair.object_b,
                "concept_id": result.concept_index + 1,
                "auto_region_label": result.auto_region_label,
                "effect_score_CE_k": result.effect_score,
                "original_score_a": result.original_score_a,
                "original_score_b": result.original_score_b,
                "original_joint_score": (
                    result.original_score_a * result.original_score_b
                ),
                "post_score_a": result.post_score_a,
                "post_score_b": result.post_score_b,
                "post_joint_score": (
                    result.post_score_a * result.post_score_b
                ),
                "mask_fraction_of_union": result.mask_fraction,
            }
        )
    pd.DataFrame(rows).to_csv(
        example_dir / "concept_scores.csv", index=False
    )

    summary = {
        "image_id": pair.image_id,
        "file_name": pair.file_name,
        "object_a": pair.object_a,
        "object_b": pair.object_b,
        "detector_score_a": detection_a.score,
        "detector_score_b": detection_b.score,
        "original_joint_score": detection_a.score * detection_b.score,
        "top_effect_score": max(
            result.effect_score for result in concept_results
        ),
        "dashboard_path": str(dashboard_path),
    }
    with open(example_dir / "summary.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)
    return summary


def run_coco_generation() -> pd.DataFrame:
    ensure_coco_files()
    selector = COCOPairSelector(
        COCO_ANNOTATION_FILE, COCO_IMAGE_DIR
    )
    processor = YOLOv5Processor()

    summaries: List[Dict[str, Any]] = []
    dashboard_paths: List[Path] = []
    used_image_ids: set[int] = set()

    for spec in PAIR_SPECS:
        if len(summaries) >= TARGET_RESULTS:
            break

        print(f"\nSearching COCO pair: {spec.object_a} + {spec.object_b}")
        candidates = selector.candidate_pairs(spec)
        successes_for_pair = 0

        for candidate in tqdm(candidates, desc=f"{spec.object_a}+{spec.object_b}"):
            if candidate.image_id in used_image_ids:
                continue
            try:
                summary = analyze_ground_truth_pair(
                    processor, candidate, selector
                )
            except Exception as error:
                print(
                    f"Skipped COCO image {candidate.image_id}: "
                    f"{type(error).__name__}: {error}"
                )
                continue

            if summary is None:
                continue

            summaries.append(summary)
            dashboard_paths.append(Path(summary["dashboard_path"]))
            used_image_ids.add(candidate.image_id)
            successes_for_pair += 1
            print(
                f"Saved result {len(summaries)}/{TARGET_RESULTS}: "
                f"COCO {candidate.image_id}, "
                f"{spec.object_a} + {spec.object_b}"
            )

            if successes_for_pair >= spec.max_examples:
                break
            if len(summaries) >= TARGET_RESULTS:
                break

    if len(summaries) < TARGET_RESULTS:
        print(
            f"\nWarning: only {len(summaries)} valid results were generated. "
            "Increase MAX_CANDIDATES_PER_PAIR, lower the initial confidence "
            "slightly, or add more PairSpec entries."
        )

    summary_frame = pd.DataFrame(summaries)
    summary_frame.to_csv(OUTPUT_DIR / "all_results_summary.csv", index=False)

    manifest = {
        "configuration": {
            "model": MODEL_NAME,
            "input_size": MODEL_INPUT_SIZE,
            "target_results": TARGET_RESULTS,
            "K": K_CONCEPTS,
            "temperature": SOFTMAX_TEMPERATURE,
            "activation_percentile": ACTIVATION_PERCENTILE,
            "blur_radius": BLUR_RADIUS,
            "initial_detection_conf": INITIAL_DETECTION_CONF,
            "intervention_detection_conf": INTERVENTION_DETECTION_CONF,
            "matching_iou_threshold": MATCH_IOU_THRESHOLD,
            "seed": SEED,
        },
        "results": summaries,
    }
    with open(OUTPUT_DIR / "run_manifest.json", "w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)

    if dashboard_paths:
        create_contact_sheet(
            dashboard_paths,
            OUTPUT_DIR / f"composite_{len(dashboard_paths)}_examples.png",
            columns=2,
        )

    print(f"\nFinished. Outputs are in: {OUTPUT_DIR}")
    return summary_frame




# ============================================================
# 9. PUBLIC-REPOSITORY BASELINE ADAPTERS
# ============================================================
# Sources used by this unified detector comparison:
# - ODAM: https://github.com/Cyang-Zhao/ODAM
# - CRAFT: https://github.com/deel-ai/Craft
# - D-RISE: https://github.com/hysts/pytorch_D-RISE
# - Finer-CAM: https://github.com/Imageomics/Finer-CAM
#
# Important scientific note:
# The original repositories use different detectors and task interfaces.
# The adapters below keep one detector (YOLOv5x6), one target instance, and
# one input image fixed. This prevents detector choice from confounding the
# qualitative comparison. The manifest labels every adaptation explicitly.

import gc
import subprocess
import time
from sklearn.decomposition import NMF
import torchvision

BASELINE_ROOT = OUTPUT_DIR / "unified_baseline_comparison"
BASELINE_ROOT.mkdir(parents=True, exist_ok=True)

# Fast Colab defaults. For final paper experiments set PAPER_MODE = True.
PAPER_MODE = False
DRISE_MASK_COUNT = 1000 if PAPER_MODE else 64
DRISE_BATCH_SIZE = 8 if PAPER_MODE else 4
DRISE_GRID_SIZE = 16
DRISE_KEEP_PROBABILITY = 0.5
CRAFT_CONCEPTS = 8 if PAPER_MODE else 6
CRAFT_EVALUATED_CONCEPTS = 6 if PAPER_MODE else 3
FINER_ALPHA = 1.0
FINER_REFERENCE_COUNT = 3
BASELINE_MATCH_IOU = 0.10

SOURCE_REPOSITORIES = {
    "ODAM": "https://github.com/Cyang-Zhao/ODAM.git",
    "CRAFT": "https://github.com/deel-ai/Craft.git",
    "D-RISE": "https://github.com/hysts/pytorch_D-RISE.git",
    "Finer-CAM": "https://github.com/Imageomics/Finer-CAM.git",
}

METHOD_SCOPE = pd.DataFrame(
    [
        {
            "method": "D-RISE",
            "native_scope": "instance-specific black-box saliency",
            "comparison_output": "one saliency map per detected object",
            "concept_discovery": False,
            "explicit_pair_relation": False,
            "adapter_status": "public PyTorch D-RISE algorithm adapted to YOLOv5x6",
        },
        {
            "method": "ODAM",
            "native_scope": "instance-specific gradient attribution",
            "comparison_output": "one gradient activation map per detected object",
            "concept_discovery": False,
            "explicit_pair_relation": False,
            "adapter_status": "official ODAM objective adapted to YOLOv5x6 raw predictions",
        },
        {
            "method": "Finer-CAM",
            "native_scope": "class-discriminative CAM",
            "comparison_output": "one class-relative map per detected object",
            "concept_discovery": False,
            "explicit_pair_relation": False,
            "adapter_status": "official weighted relative target adapted to YOLOv5x6",
        },
        {
            "method": "CRAFT",
            "native_scope": "concept discovery and concept importance",
            "comparison_output": "object-conditioned concept maps",
            "concept_discovery": True,
            "explicit_pair_relation": False,
            "adapter_status": "CRAFT NMF concept core adapted to detector feature maps",
        },
        {
            "method": "RCE (ours)",
            "native_scope": "relational concept explanation",
            "comparison_output": "pair-conditioned concepts and joint intervention scores",
            "concept_discovery": True,
            "explicit_pair_relation": True,
            "adapter_status": "user-provided New_COCO_Experi model",
        },
    ]
)
METHOD_SCOPE.to_csv(BASELINE_ROOT / "method_scope.csv", index=False)


def clone_public_sources(destination: Path = Path("/content/public_xai_sources")) -> Dict[str, str]:
    """Clone public reference repositories and return their commit hashes.

    The unified adapters are self-contained so that a temporary GitHub outage does
    not invalidate an already-open Colab runtime. Cloning records exact provenance.
    """
    destination.mkdir(parents=True, exist_ok=True)
    commits: Dict[str, str] = {}
    for name, url in SOURCE_REPOSITORIES.items():
        repo_dir = destination / name.replace("/", "_")
        if not repo_dir.exists():
            completed = subprocess.run(
                ["git", "clone", "--depth", "1", url, str(repo_dir)],
                text=True,
                capture_output=True,
            )
            if completed.returncode != 0:
                print(f"[Source warning] {name} clone failed: {completed.stderr.strip()}")
                commits[name] = "clone_failed"
                continue
        completed = subprocess.run(
            ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
            text=True,
            capture_output=True,
        )
        commits[name] = (
            completed.stdout.strip() if completed.returncode == 0 else "unknown"
        )
    with open(BASELINE_ROOT / "source_commits.json", "w", encoding="utf-8") as file:
        json.dump({"repositories": SOURCE_REPOSITORIES, "commits": commits}, file, indent=2)
    return commits


def normalize01(values: np.ndarray, support: Optional[np.ndarray] = None) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    result = np.zeros_like(values, dtype=np.float32)
    if support is None:
        valid = np.isfinite(values)
    else:
        valid = (np.asarray(support) > 0) & np.isfinite(values)
    if not np.any(valid):
        return result
    lo = float(values[valid].min())
    hi = float(values[valid].max())
    if hi - lo <= 1e-8:
        return result
    result[valid] = (values[valid] - lo) / (hi - lo)
    return result


def matched_detection_score(
    detections: Sequence[Detection],
    target: Detection,
    minimum_iou: float = BASELINE_MATCH_IOU,
) -> float:
    best = 0.0
    for detection in detections:
        if detection.label_idx != target.label_idx:
            continue
        overlap = calculate_iou(target.box_xyxy, detection.box_xyxy)
        if overlap >= minimum_iou:
            best = max(best, float(detection.score))
    return best


def matched_detection_similarity(
    detections: Sequence[Detection], target: Detection
) -> float:
    """D-RISE localization × categorization similarity.

    YOLOv5 post-NMS outputs expose the predicted class confidence rather than the
    full class vector. The category similarity therefore becomes target-class
    confidence for detections of the same class, multiplied by box IoU.
    """
    best = 0.0
    for detection in detections:
        if detection.label_idx != target.label_idx:
            continue
        overlap = calculate_iou(target.box_xyxy, detection.box_xyxy)
        best = max(best, overlap * float(detection.score))
    return best


def detect_batch(
    processor: YOLOv5Processor,
    images: Sequence[Image.Image],
    confidence: float = INTERVENTION_DETECTION_CONF,
) -> List[List[Detection]]:
    old_conf = float(processor.detector.conf)
    processor.detector.conf = float(confidence)
    try:
        with torch.no_grad():
            results = processor.detector(list(images), size=processor.input_size)
        frames = results.pandas().xyxy
        all_detections: List[List[Detection]] = []
        for frame in frames:
            detections: List[Detection] = []
            for _, row in frame.iterrows():
                label_idx = int(row["class"])
                detections.append(
                    Detection(
                        box_xyxy=np.array(
                            [row.xmin, row.ymin, row.xmax, row.ymax],
                            dtype=np.float32,
                        ),
                        score=float(row.confidence),
                        label_idx=label_idx,
                        class_name=str(processor.detector.names[label_idx]),
                    )
                )
            all_detections.append(detections)
        return all_detections
    finally:
        processor.detector.conf = old_conf


def random_drise_masks(
    count: int,
    height: int,
    width: int,
    grid_size: int = DRISE_GRID_SIZE,
    keep_probability: float = DRISE_KEEP_PROBABILITY,
    seed: int = SEED,
) -> Iterable[np.ndarray]:
    """Generate RISE/D-RISE masks using a coarse Bernoulli grid and random shift."""
    rng = np.random.default_rng(seed)
    cell_h = int(math.ceil(height / grid_size))
    cell_w = int(math.ceil(width / grid_size))
    up_h = (grid_size + 1) * cell_h
    up_w = (grid_size + 1) * cell_w
    for _ in range(count):
        coarse = (rng.random((grid_size, grid_size)) < keep_probability).astype(np.float32)
        upsampled = cv2.resize(coarse, (up_w, up_h), interpolation=cv2.INTER_LINEAR)
        shift_y = int(rng.integers(0, cell_h))
        shift_x = int(rng.integers(0, cell_w))
        yield upsampled[shift_y : shift_y + height, shift_x : shift_x + width]


def drise_pair_maps(
    processor: YOLOv5Processor,
    image: Image.Image,
    detection_a: Detection,
    detection_b: Detection,
    mask_count: int = DRISE_MASK_COUNT,
    batch_size: int = DRISE_BATCH_SIZE,
) -> Tuple[np.ndarray, np.ndarray]:
    """Create two independent D-RISE maps while sharing the same random masks."""
    image_rgb = np.asarray(image.convert("RGB"), dtype=np.float32)
    height, width = image_rgb.shape[:2]
    saliency_a = np.zeros((height, width), dtype=np.float64)
    saliency_b = np.zeros((height, width), dtype=np.float64)
    masks = list(random_drise_masks(mask_count, height, width))

    for start in tqdm(range(0, mask_count, batch_size), desc="D-RISE masks", leave=False):
        batch_masks = masks[start : start + batch_size]
        masked_images = [
            Image.fromarray(np.uint8(np.clip(image_rgb * mask[..., None], 0, 255)))
            for mask in batch_masks
        ]
        batch_detections = detect_batch(processor, masked_images)
        for mask, detections in zip(batch_masks, batch_detections):
            saliency_a += matched_detection_similarity(detections, detection_a) * mask
            saliency_b += matched_detection_similarity(detections, detection_b) * mask

    denominator = float(mask_count) * float(DRISE_KEEP_PROBABILITY) + 1e-8
    return normalize01(saliency_a / denominator), normalize01(saliency_b / denominator)


def _find_prediction_tensor(value: Any) -> Optional[torch.Tensor]:
    if torch.is_tensor(value) and value.ndim == 3 and value.shape[-1] >= 6:
        return value
    if isinstance(value, (list, tuple)):
        for item in value:
            found = _find_prediction_tensor(item)
            if found is not None:
                return found
    if isinstance(value, dict):
        for item in value.values():
            found = _find_prediction_tensor(item)
            if found is not None:
                return found
    return None


def xywh_tensor_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    x, y, w, h = boxes.unbind(dim=-1)
    return torch.stack((x - w / 2, y - h / 2, x + w / 2, y + h / 2), dim=-1)


def map_box_to_letterbox_tensor(box: np.ndarray, meta: LetterboxMeta, device: torch.device) -> torch.Tensor:
    transformed = torch.tensor(
        [
            box[0] * meta.ratio + meta.left,
            box[1] * meta.ratio + meta.top,
            box[2] * meta.ratio + meta.left,
            box[3] * meta.ratio + meta.top,
        ],
        dtype=torch.float32,
        device=device,
    )
    return transformed


class YOLOv5GradientAdapter:
    """Gradient bridge for ODAM and Finer-CAM on the same YOLOv5x6 detector."""

    def __init__(self, processor: YOLOv5Processor) -> None:
        self.processor = processor
        self.backend = processor.backend
        self.detect_layer = processor.detect_layer

    def _prepare(self, image: Image.Image) -> Tuple[torch.Tensor, LetterboxMeta]:
        image_rgb = np.asarray(image.convert("RGB"))
        padded, meta = letterbox_rgb(
            image_rgb, new_shape=(self.processor.input_size, self.processor.input_size)
        )
        tensor = (
            torch.from_numpy(padded)
            .to(self.processor.device)
            .permute(2, 0, 1)
            .contiguous()
            .unsqueeze(0)
            .float()
            / 255.0
        )
        if getattr(self.backend, "fp16", False):
            tensor = tensor.half()
        tensor.requires_grad_(True)
        return tensor, meta

    def _forward(self, image: Image.Image) -> Tuple[torch.Tensor, List[torch.Tensor], LetterboxMeta]:
        tensor, meta = self._prepare(image)
        captured: List[torch.Tensor] = []

        def capture_inputs(module: torch.nn.Module, inputs: Tuple[Any, ...]) -> None:
            tensors: List[torch.Tensor] = []
            _collect_4d_tensors(inputs, tensors)
            captured.extend(tensors)
            for feature in tensors:
                if feature.requires_grad:
                    feature.retain_grad()

        handle = self.detect_layer.register_forward_pre_hook(capture_inputs)
        try:
            output = self.backend(tensor, augment=False, visualize=False)
        finally:
            handle.remove()
        prediction = _find_prediction_tensor(output)
        if prediction is None:
            raise RuntimeError("Could not locate YOLOv5 raw prediction tensor.")
        if not captured:
            raise RuntimeError("Could not capture YOLOv5 multi-scale detector features.")
        return prediction, captured, meta

    def _select_anchor(
        self,
        prediction: torch.Tensor,
        target: Detection,
        meta: LetterboxMeta,
    ) -> Tuple[int, torch.Tensor, torch.Tensor]:
        candidates = prediction[0]
        class_index = 5 + int(target.label_idx)
        if class_index >= candidates.shape[1]:
            raise IndexError(f"Class index {target.label_idx} is outside prediction tensor.")
        boxes = xywh_tensor_to_xyxy(candidates[:, :4].float())
        target_box = map_box_to_letterbox_tensor(
            target.box_xyxy, meta, candidates.device
        ).unsqueeze(0)
        overlaps = torchvision.ops.box_iou(target_box, boxes)[0]
        class_score = candidates[:, 4].float() * candidates[:, class_index].float()
        quality = overlaps * class_score
        selected_index = int(torch.argmax(quality).item())
        return selected_index, overlaps[selected_index], class_score[selected_index]

    def _aggregate_feature_maps(
        self,
        features: Sequence[torch.Tensor],
        meta: LetterboxMeta,
        method: str,
    ) -> np.ndarray:
        maps: List[np.ndarray] = []
        for feature in features:
            gradient = feature.grad
            if gradient is None:
                continue
            activation = feature.float()
            gradient = gradient.float()
            if float(gradient.detach().abs().sum()) <= 1e-12:
                continue
            if method == "odam":
                local_map = torch.relu((activation * gradient).sum(dim=1))[0]
            elif method == "finer_cam":
                weights = gradient.mean(dim=(2, 3), keepdim=True)
                local_map = torch.relu((activation * weights).sum(dim=1))[0]
            else:
                raise ValueError(f"Unknown gradient map method: {method}")
            original_map = feature_map_to_original(
                local_map.detach().cpu().numpy(), meta
            )
            maps.append(normalize01(original_map))
        if not maps:
            original_w, original_h = meta.original_size
            return np.zeros((original_h, original_w), dtype=np.float32)
        stacked = np.stack(maps, axis=0)
        merged = np.max(stacked, axis=0)
        merged = cv2.GaussianBlur(merged, (0, 0), sigmaX=1.2)
        return normalize01(merged)

    def odam(self, image: Image.Image, target: Detection) -> Tuple[np.ndarray, Dict[str, Any]]:
        self.backend.zero_grad(set_to_none=True)
        prediction, features, meta = self._forward(image)
        index, overlap, anchor_score = self._select_anchor(prediction, target, meta)
        selected = prediction[0, index]
        objective = selected[4].float() * selected[5 + target.label_idx].float()
        objective.backward(retain_graph=False)
        heatmap = self._aggregate_feature_maps(features, meta, method="odam")
        return heatmap, {
            "selected_anchor": index,
            "anchor_iou": float(overlap.detach().cpu()),
            "anchor_class_score": float(anchor_score.detach().cpu()),
        }

    def finer_cam(
        self,
        image: Image.Image,
        target: Detection,
        alpha: float = FINER_ALPHA,
        reference_count: int = FINER_REFERENCE_COUNT,
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        self.backend.zero_grad(set_to_none=True)
        prediction, features, meta = self._forward(image)
        index, overlap, anchor_score = self._select_anchor(prediction, target, meta)
        selected = prediction[0, index]
        class_probabilities = selected[5:].float().clamp(1e-6, 1.0 - 1e-6)
        class_logits = torch.logit(class_probabilities)
        main_index = int(target.label_idx)
        differences = torch.abs(class_logits - class_logits[main_index])
        differences[main_index] = torch.inf
        references = torch.argsort(differences)[:reference_count]
        reference_weights = torch.softmax(class_logits[references], dim=0)
        relative_objective = (
            reference_weights
            * (class_logits[main_index] - float(alpha) * class_logits[references])
        ).sum() / (reference_weights.sum() + 1e-9)
        objective = selected[4].float() * relative_objective
        objective.backward(retain_graph=False)
        heatmap = self._aggregate_feature_maps(features, meta, method="finer_cam")
        reference_names = [
            str(self.processor.detector.names[int(index_value)])
            for index_value in references.detach().cpu().tolist()
        ]
        return heatmap, {
            "selected_anchor": index,
            "anchor_iou": float(overlap.detach().cpu()),
            "anchor_class_score": float(anchor_score.detach().cpu()),
            "reference_classes": reference_names,
        }


@dataclass
class CraftConcept:
    concept_index: int
    importance: float
    activation_mass: float
    activation_map: np.ndarray


def craft_object_concepts(
    processor: YOLOv5Processor,
    image: Image.Image,
    target: Detection,
    feature_tensor: torch.Tensor,
    meta: LetterboxMeta,
    number_of_concepts: int = CRAFT_CONCEPTS,
    evaluated_concepts: int = CRAFT_EVALUATED_CONCEPTS,
) -> Tuple[np.ndarray, List[CraftConcept]]:
    """CRAFT-style NMF concepts for one object, without pair conditioning.

    CRAFT's official repository targets classifiers. This detector adapter keeps its
    non-negative matrix factorization concept core, then estimates object-specific
    importance by detector confidence drop after blurring a concept region. Unlike
    RCE, each object is explained independently and no joint pair score is used.
    """
    _, channels, feature_h, feature_w = feature_tensor.shape
    x1, y1, x2, y2 = map_box_to_feature(target.box_xyxy, meta, (feature_h, feature_w))
    if x2 <= x1 or y2 <= y1:
        return np.zeros((image.height, image.width), dtype=np.float32), []

    feature_grid = feature_tensor[0].permute(1, 2, 0).detach().float().cpu().numpy()
    feature_grid = np.clip(feature_grid, 0.0, None)
    region_vectors = feature_grid[y1:y2, x1:x2].reshape(-1, channels)
    valid = np.isfinite(region_vectors).all(axis=1)
    region_vectors = region_vectors[valid]
    if region_vectors.shape[0] < 2 or float(region_vectors.max()) <= 0.0:
        return np.zeros((image.height, image.width), dtype=np.float32), []

    component_count = max(
        1, min(int(number_of_concepts), region_vectors.shape[0], region_vectors.shape[1])
    )
    factorizer = NMF(
        n_components=component_count,
        init="nndsvda",
        random_state=SEED,
        max_iter=500,
    )
    factorizer.fit(region_vectors + 1e-8)
    all_vectors = feature_grid.reshape(-1, channels)
    concept_coefficients = factorizer.transform(all_vectors + 1e-8)
    concept_maps = concept_coefficients.reshape(feature_h, feature_w, component_count)

    object_support = box_mask(target.box_xyxy, image.size)
    candidates: List[Tuple[int, float, np.ndarray]] = []
    for concept_index in range(component_count):
        original_map = feature_map_to_original(concept_maps[..., concept_index], meta)
        normalized = normalize01(original_map, object_support)
        activation_mass = float((original_map * object_support).sum())
        candidates.append((concept_index, activation_mass, normalized))
    candidates.sort(key=lambda item: item[1], reverse=True)

    image_rgb = np.asarray(image.convert("RGB"))
    blurred = np.asarray(image.filter(ImageFilter.GaussianBlur(radius=BLUR_RADIUS)))
    concepts: List[CraftConcept] = []
    for concept_index, activation_mass, normalized in candidates[:evaluated_concepts]:
        values = normalized[object_support.astype(bool)]
        if values.size == 0 or float(values.max()) <= 0.0:
            importance = 0.0
        else:
            threshold = float(np.percentile(values, ACTIVATION_PERCENTILE))
            mask = ((normalized >= threshold) & (object_support > 0)).astype(np.uint8)
            counterfactual = np.where(mask[..., None] > 0, blurred, image_rgb).astype(np.uint8)
            detections = processor.detect(
                Image.fromarray(counterfactual),
                conf_threshold=INTERVENTION_DETECTION_CONF,
            )
            post_score = matched_detection_score(detections, target)
            importance = max(0.0, float(target.score) - post_score)
        concepts.append(
            CraftConcept(
                concept_index=int(concept_index),
                importance=float(importance),
                activation_mass=float(activation_mass),
                activation_map=normalized,
            )
        )

    if not concepts:
        return np.zeros((image.height, image.width), dtype=np.float32), []
    weights = np.array([concept.importance for concept in concepts], dtype=np.float32)
    if float(weights.sum()) <= 1e-8:
        weights = np.array([concept.activation_mass for concept in concepts], dtype=np.float32)
    weights = weights / (float(weights.sum()) + 1e-8)
    combined = np.zeros((image.height, image.width), dtype=np.float32)
    for weight, concept in zip(weights, concepts):
        combined += float(weight) * concept.activation_map
    return normalize01(combined, object_support), concepts


@dataclass
class UnifiedBaselineResult:
    maps: Dict[str, np.ndarray]
    metadata: Dict[str, Any]
    runtimes: Dict[str, float]


def compute_unified_baselines(
    processor: YOLOv5Processor,
    image: Image.Image,
    detection_a: Detection,
    detection_b: Detection,
    feature_tensor: torch.Tensor,
    feature_meta: LetterboxMeta,
) -> UnifiedBaselineResult:
    maps: Dict[str, np.ndarray] = {}
    metadata: Dict[str, Any] = {}
    runtimes: Dict[str, float] = {}

    start = time.perf_counter()
    maps["drise_a"], maps["drise_b"] = drise_pair_maps(
        processor, image, detection_a, detection_b
    )
    runtimes["D-RISE"] = time.perf_counter() - start

    gradient_adapter = YOLOv5GradientAdapter(processor)

    start = time.perf_counter()
    maps["odam_a"], metadata["odam_a"] = gradient_adapter.odam(image, detection_a)
    maps["odam_b"], metadata["odam_b"] = gradient_adapter.odam(image, detection_b)
    runtimes["ODAM"] = time.perf_counter() - start
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    start = time.perf_counter()
    maps["finer_a"], metadata["finer_a"] = gradient_adapter.finer_cam(image, detection_a)
    maps["finer_b"], metadata["finer_b"] = gradient_adapter.finer_cam(image, detection_b)
    runtimes["Finer-CAM"] = time.perf_counter() - start
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    start = time.perf_counter()
    maps["craft_a"], craft_a = craft_object_concepts(
        processor, image, detection_a, feature_tensor, feature_meta
    )
    maps["craft_b"], craft_b = craft_object_concepts(
        processor, image, detection_b, feature_tensor, feature_meta
    )
    runtimes["CRAFT-adapted"] = time.perf_counter() - start
    metadata["craft_a"] = [
        {
            "concept": concept.concept_index + 1,
            "importance": concept.importance,
            "activation_mass": concept.activation_mass,
        }
        for concept in craft_a
    ]
    metadata["craft_b"] = [
        {
            "concept": concept.concept_index + 1,
            "importance": concept.importance,
            "activation_mass": concept.activation_mass,
        }
        for concept in craft_b
    ]

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return UnifiedBaselineResult(maps=maps, metadata=metadata, runtimes=runtimes)


def save_map_arrays(maps: Dict[str, np.ndarray], destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    for name, values in maps.items():
        np.save(destination / f"{name}.npy", np.asarray(values, dtype=np.float32))


def create_unified_comparison_figure(
    image: Image.Image,
    image_id: int,
    detection_a: Detection,
    detection_b: Detection,
    pair_union: np.ndarray,
    baseline_result: UnifiedBaselineResult,
    rce_concepts: Sequence[ConceptResult],
    output_stem: Path,
) -> Path:
    image_rgb = np.asarray(image.convert("RGB"))
    boxed = image_rgb.copy()
    draw_labelled_box(boxed, detection_a, (0, 255, 255))
    draw_labelled_box(boxed, detection_b, (255, 255, 0))
    union_view = make_union_overlay(image_rgb, pair_union)
    rce_map = combined_top_concept_heatmap(rce_concepts, pair_union, image.size, top_n=3)

    maps = baseline_result.maps
    fig, axes = plt.subplots(2, 6, figsize=(24, 9.2))
    fig.suptitle(
        f"Unified XAI Baseline Comparison — {detection_a.class_name} + "
        f"{detection_b.class_name} | COCO {image_id}",
        fontsize=18,
        fontweight="bold",
    )

    axes[0, 0].imshow(boxed)
    axes[0, 0].set_title("Target object pair", fontweight="bold")
    axes[1, 0].imshow(union_view)
    axes[1, 0].set_title("RCE relational union", fontweight="bold")

    panel_specs = [
        (1, "D-RISE", "drise_a", "drise_b"),
        (2, "ODAM", "odam_a", "odam_b"),
        (3, "Finer-CAM", "finer_a", "finer_b"),
        (4, "CRAFT-adapted", "craft_a", "craft_b"),
    ]
    for column, method, key_a, key_b in panel_specs:
        axes[0, column].imshow(overlay_heatmap(image_rgb, maps[key_a]))
        axes[0, column].set_title(
            f"{method}: {detection_a.class_name}", fontweight="bold"
        )
        axes[1, column].imshow(overlay_heatmap(image_rgb, maps[key_b]))
        axes[1, column].set_title(
            f"{method}: {detection_b.class_name}", fontweight="bold"
        )

    axes[0, 5].imshow(overlay_heatmap(image_rgb, rce_map))
    axes[0, 5].set_title("RCE: relational concepts", fontweight="bold")

    ranked = sorted(rce_concepts, key=lambda result: result.effect_score, reverse=True)
    displayed = ranked[: min(SHOW_TOP_CONCEPTS, len(ranked))]
    labels = [f"C{item.concept_index + 1}: {item.auto_region_label}" for item in displayed]
    scores = [item.effect_score for item in displayed]
    positions = np.arange(len(labels))
    axes[1, 5].barh(positions, scores, edgecolor="black", linewidth=0.8)
    axes[1, 5].set_yticks(positions)
    axes[1, 5].set_yticklabels(labels, fontsize=8)
    axes[1, 5].invert_yaxis()
    axes[1, 5].set_xlabel(r"Joint concept effect $CE_k=\min(\Delta s_A,\Delta s_B)$")
    axes[1, 5].set_title("RCE: joint intervention ranking", fontweight="bold")
    axes[1, 5].grid(axis="x", linestyle=":", alpha=0.35)

    for axis in axes.flat:
        if axis is not axes[1, 5]:
            axis.axis("off")

    fig.text(
        0.5,
        0.012,
        "D-RISE, ODAM and Finer-CAM explain each detection independently; "
        "CRAFT discovers object-conditioned concepts; RCE discovers concepts in the "
        "pair union and ranks their joint causal influence on both detections.",
        ha="center",
        fontsize=11,
    )
    plt.tight_layout(rect=[0.0, 0.04, 1.0, 0.95])
    png_path = output_stem.with_suffix(".png")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    if SAVE_PDF:
        fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    return png_path


def analyze_pair_with_all_baselines(
    processor: YOLOv5Processor,
    pair: GroundTruthPair,
    selector: COCOPairSelector,
) -> Optional[Dict[str, Any]]:
    image_path = selector.image_path(pair)
    if not image_path.exists():
        return None
    image = Image.open(image_path).convert("RGB")
    image_area = float(image.width * image.height)
    detections = processor.detect(image, conf_threshold=INITIAL_DETECTION_CONF)
    detection_a = match_detection_to_ground_truth(
        detections, pair.object_a, pair.box_a, minimum_iou=BASELINE_MATCH_IOU
    )
    detection_b = match_detection_to_ground_truth(
        detections, pair.object_b, pair.box_b, minimum_iou=BASELINE_MATCH_IOU
    )
    if detection_a is None or detection_b is None:
        return None
    if box_area(detection_a.box_xyxy) < image_area * 0.04:
        return None
    if box_area(detection_b.box_xyxy) < image_area * 0.03:
        return None

    pair_union = union_box(detection_a.box_xyxy, detection_b.box_xyxy)
    feature_tensor, feature_meta = processor.extract_features(image)

    rce_start = time.perf_counter()
    activation_maps = KMeansConceptExplainer().discover(
        feature_tensor, pair_union, feature_meta
    )
    rce_results = RelationalInterventionExplainer(
        processor, matching_iou=BASELINE_MATCH_IOU
    ).evaluate(
        image=image,
        activation_maps=activation_maps,
        meta=feature_meta,
        detection_a=detection_a,
        detection_b=detection_b,
        pair_union=pair_union,
    )
    rce_runtime = time.perf_counter() - rce_start

    baseline_result = compute_unified_baselines(
        processor,
        image,
        detection_a,
        detection_b,
        feature_tensor,
        feature_meta,
    )
    baseline_result.runtimes["RCE (ours)"] = rce_runtime

    stem = (
        f"coco_{pair.image_id}_{pair.object_a.replace(' ', '_')}_"
        f"{pair.object_b.replace(' ', '_')}"
    )
    example_dir = BASELINE_ROOT / stem
    example_dir.mkdir(parents=True, exist_ok=True)
    save_map_arrays(baseline_result.maps, example_dir / "maps")
    rce_map = combined_top_concept_heatmap(rce_results, pair_union, image.size, top_n=3)
    np.save(example_dir / "maps" / "rce_relational.npy", rce_map.astype(np.float32))

    figure_path = create_unified_comparison_figure(
        image=image,
        image_id=pair.image_id,
        detection_a=detection_a,
        detection_b=detection_b,
        pair_union=pair_union,
        baseline_result=baseline_result,
        rce_concepts=rce_results,
        output_stem=example_dir / "unified_comparison",
    )

    rce_rows = [
        {
            "concept": item.concept_index + 1,
            "region_label": item.auto_region_label,
            "CE_k": item.effect_score,
            "score_drop_a": item.original_score_a - item.post_score_a,
            "score_drop_b": item.original_score_b - item.post_score_b,
            "mask_fraction": item.mask_fraction,
        }
        for item in sorted(rce_results, key=lambda value: value.effect_score, reverse=True)
    ]
    pd.DataFrame(rce_rows).to_csv(example_dir / "rce_concept_scores.csv", index=False)
    pd.DataFrame(
        [{"method": method, "seconds": seconds} for method, seconds in baseline_result.runtimes.items()]
    ).to_csv(example_dir / "runtime.csv", index=False)

    metadata = {
        "image_id": pair.image_id,
        "file_name": pair.file_name,
        "object_a": asdict(detection_a),
        "object_b": asdict(detection_b),
        "pair_union": pair_union.tolist(),
        "baseline_metadata": baseline_result.metadata,
        "runtimes": baseline_result.runtimes,
        "figure_path": str(figure_path),
    }
    # Convert NumPy arrays in dataclass dictionaries to JSON-safe lists.
    metadata["object_a"]["box_xyxy"] = detection_a.box_xyxy.tolist()
    metadata["object_b"]["box_xyxy"] = detection_b.box_xyxy.tolist()
    with open(example_dir / "metadata.json", "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2)

    return {
        "image_id": pair.image_id,
        "object_a": pair.object_a,
        "object_b": pair.object_b,
        "score_a": detection_a.score,
        "score_b": detection_b.score,
        "top_RCE_CE_k": max(item.effect_score for item in rce_results),
        "comparison_figure": str(figure_path),
        **{f"runtime_{key}": value for key, value in baseline_result.runtimes.items()},
    }


def run_full_baseline_comparison() -> pd.DataFrame:
    print(METHOD_SCOPE.to_string(index=False))
    clone_public_sources()
    ensure_coco_files()
    selector = COCOPairSelector(COCO_ANNOTATION_FILE, COCO_IMAGE_DIR)
    processor = YOLOv5Processor(
        model_name=MODEL_NAME,
        device=DEVICE,
        input_size=MODEL_INPUT_SIZE,
    )

    summaries: List[Dict[str, Any]] = []
    figures: List[Path] = []
    used_ids: set[int] = set()
    for spec in PAIR_SPECS:
        if len(summaries) >= TARGET_RESULTS:
            break
        print(f"\nSearching: {spec.object_a} + {spec.object_b}")
        successful_for_pair = 0
        for candidate in selector.candidate_pairs(spec):
            if candidate.image_id in used_ids:
                continue
            try:
                summary = analyze_pair_with_all_baselines(processor, candidate, selector)
            except torch.cuda.OutOfMemoryError:
                print(f"Skipped {candidate.image_id}: CUDA out of memory.")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            except Exception as error:
                print(f"Skipped {candidate.image_id}: {type(error).__name__}: {error}")
                continue
            if summary is None:
                continue
            summaries.append(summary)
            figures.append(Path(summary["comparison_figure"]))
            used_ids.add(candidate.image_id)
            successful_for_pair += 1
            print(
                f"Saved {len(summaries)}/{TARGET_RESULTS}: COCO {candidate.image_id} "
                f"({spec.object_a}+{spec.object_b})"
            )
            if successful_for_pair >= spec.max_examples or len(summaries) >= TARGET_RESULTS:
                break

    result_frame = pd.DataFrame(summaries)
    result_frame.to_csv(BASELINE_ROOT / "all_comparison_results.csv", index=False)
    if figures:
        create_contact_sheet(
            figures,
            BASELINE_ROOT / f"all_{len(figures)}_comparison_examples.png",
            columns=1,
            panel_width=2400,
        )
    manifest = {
        "detector": MODEL_NAME,
        "input_size": MODEL_INPUT_SIZE,
        "paper_mode": PAPER_MODE,
        "D_RISE_masks": DRISE_MASK_COUNT,
        "target_results": TARGET_RESULTS,
        "method_scope": METHOD_SCOPE.to_dict(orient="records"),
        "results": summaries,
    }
    with open(BASELINE_ROOT / "comparison_manifest.json", "w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)
    print(f"\nFinished. Results: {BASELINE_ROOT}")
    return result_frame


# Colab execution
comparison_results = run_full_baseline_comparison()
print(comparison_results.to_string(index=False))

# Create one downloadable archive.
archive_path = shutil.make_archive(
    "/content/RCE_Unified_Baseline_Comparison",
    "zip",
    root_dir=BASELINE_ROOT,
)
print(f"\nDownload archive: {archive_path}")
try:
    from google.colab import files as colab_files
    colab_files.download(archive_path)
except Exception:
    pass


    method                             native_scope                                       comparison_output  concept_discovery  explicit_pair_relation                                              adapter_status
    D-RISE     instance-specific black-box saliency                    one saliency map per detected object              False                   False         public PyTorch D-RISE algorithm adapted to YOLOv5x6
      ODAM   instance-specific gradient attribution         one gradient activation map per detected object              False                   False official ODAM objective adapted to YOLOv5x6 raw predictions
 Finer-CAM                 class-discriminative CAM              one class-relative map per detected object              False                   False       official weighted relative target adapted to YOLOv5x6
     CRAFT concept discovery and concept importance                         object-conditioned concept maps               True                   False     C

Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-7-19 Python-3.12.13 torch-2.11.0+cpu CPU

Fusing layers... 
YOLOv5x6 summary: 574 layers, 140730220 parameters, 0 gradients, 209.6 GFLOPs
Adding AutoShape... 



Searching: person + laptop


D-RISE masks:   0%|          | 0/16 [00:00<?, ?it/s]

Saved 1/2: COCO 451155 (person+laptop)


D-RISE masks:   0%|          | 0/16 [00:00<?, ?it/s]

Saved 2/2: COCO 367095 (person+laptop)

Finished. Results: /content/rce_baseline_comparison_outputs/unified_baseline_comparison
 image_id object_a object_b  score_a  score_b  top_RCE_CE_k                                                                                                     comparison_figure  runtime_D-RISE  runtime_ODAM  runtime_Finer-CAM  runtime_CRAFT-adapted  runtime_RCE (ours)
   451155   person   laptop 0.434239 0.748113      0.095594 /content/rce_baseline_comparison_outputs/unified_baseline_comparison/coco_451155_person_laptop/unified_comparison.png      199.796384     17.045678          15.867898              19.101187           19.707929
   367095   person   laptop 0.940816 0.876139      0.494758 /content/rce_baseline_comparison_outputs/unified_baseline_comparison/coco_367095_person_laptop/unified_comparison.png      161.349087     15.441454          16.588329              15.955914           16.487084

Download archive: /content/RCE_Unified_Baseline_Comparison.zi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>